# Change to ur dropbox path as needed!

In [18]:
using LinearAlgebra
caliendo_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Caliendo_2019/"
caliendo_path_base_year = caliendo_path  * "Base_Year/"



"/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Caliendo_2019/Base_Year/"

In [19]:
caliendo_path

"/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Caliendo_2019/"

# data.m

In [20]:
function data(J, JNT, JT, R, N, C, B_usa, B_row, gamma, IO_data, GO, xbilat)
    # share of capital structures in value added
    B_aux = [B_usa; B_row]
    B = ones(J, 1) * B_aux'

    # share of value added in gross production
    gamma = gamma'

    # input-output table
    IO_US = IO_data[1:J, :]
    IO_US = repeat(IO_US, outer=(R, 1))
    IO_ROW = IO_data[J+1:J*(C+1), :]
    IO = [IO_US; IO_ROW]

    # share of intermediate inputs in gross outputs
    gamma_aux = (1 .- gamma')
    gamma_aux2 = zeros(J*N, J)
    G_IO = zeros(J*N, J)

    for n = 1:N
        G_IO[1+(n-1)*J:n*J, 1:J] = (kron((1 .- gamma[:, n])', ones(J, 1))) .* IO[1+(n-1)*J:n*J, 1:J]
    end
    G = G_IO

    # Calculating GO except for domestic sales across regions
    E = zeros(J, N)
    for j = 1:J
        for n = 1:N
            E[j, n] = sum(xbilat[1+N*(j-1):N*j, n])'
        end
    end
    GO_aux = E
    GO = GO'
    DS_nt_usa = GO[JT+1:J, 1:R] .- GO_aux[JT+1:J, 1:R]
    DS_nt_usa = DS_nt_usa'
    xbilat_nt_usa = zeros(R*JNT, R)
    
    for j = 1:JNT
        xbilat_nt_usa[1+(j-1)*R:j*R, :] = Diagonal(DS_nt_usa[:, j])
    end

    xbilat_nt = xbilat[JT*N+1:J*N, :]
    for j = 1:JNT
        xbilat_nt[1+(j-1)*N:N*j-C, 1:R] = xbilat_nt_usa[1+(j-1)*R:j*R, :]
    end
    xbilat = [xbilat[1:JT*N, :]; xbilat_nt]

    # Check GO
    for j = 1:J
        for n = 1:N
            E[j, n] = sum(xbilat[1+N*(j-1):N*j, n])'
        end
    end
    GO_check = E
    GO = GO_check

    # Calculating Din
    Xjn = sum(xbilat', dims=1)' * ones(1, N)
    Din = xbilat ./ Xjn

    # Calculating aggregate deficits
    M = zeros(J, N)
    for j = 1:J
        M[j, :] = sum(xbilat[1+N*(j-1):N*j, :], dims=2)'
        for n = 1:N
            E[j, n] = sum(xbilat[1+N*(j-1):N*j, n])'
        end
    end
    Bn = sum(E, dims=1)' .- sum(M, dims=1)'

    # Calculating X0
    A = sum(xbilat', dims=1)
    X0 = zeros(J, N)
    for j = 1:J
        X0[j, :] = A[:, 1+N*(j-1):N*j]
    end

    # Calculating labor compensation, payment to structures and value added
    PQ_vec0 = reshape(X0', J*N, 1)
    DP0 = zeros(J*N, N)
    for n = 1:N
        DP0[:, n] = Din[:, n] .* PQ_vec0
    end
    
    Exjn0 = zeros(J, N)
    for j = 1:J
        for n = 1:N
            Exjn0[j, n] = sum(DP0[1+N*(j-1):N*j, n])'
        end
    end
    
    VALjn0 = gamma .* (1 .- B) .* Exjn0
    VARjn0 = (B ./ (1 .- B)) .* VALjn0
    VAL = sum(VALjn0, dims=1)'
    VAR = sum(VARjn0, dims=1)'
    VA = VAR .+ VAL

    # alphas
    aux2 = zeros(J, N)
    for n = 1:N
        irow = 1+J*(n-1):J*n
        aux2[:, n] = X0[:, n] .- G[irow, :] * E[:, n]
    end
    
    alphas = (sum(aux2', dims=1)' * ones(1, N)) ./ sum(VA .- Bn)

    # Iotas
    Chi = sum(VAR)
    io = (VAR .- Bn) ./ Chi
    Sn = Bn .- VAR .+ io .* Chi

    return B, gamma, G, Din, VALjn0, VARjn0, alphas, VAR, Bn
end

data (generic function with 1 method)

# P_h_om.m

In [21]:
function P_h_om(om_n, kappa_hat, lambda_hat, T, B, G, gamma, Din, J, N, maxit, tol)
    # Initialize vectors of ex-post factor and good prices
    om = om_n
    pf0 = ones(J, N)
    
    pfmax = 1
    it = 1
    
    while (it <= maxit) && (pfmax > tol)
        lom = log.(om)
        lp = log.(pf0)
        
        # Calculating input bundle costs
        lc = zeros(J, N)
        for i = 1:N
            lc[:, i] = gamma[:, i] .* lom[:, i] .+ (G[1+(i-1)*J:J*i, :]' * lp[:, i])
        end
        c = exp.(lc)
        
        LT = zeros(J*N, 1)
        for j = 1:J
            idx = 1+(j-1)*N:N*j
            LT[idx, 1] = ones(N) * T[j]
        end
        Din_k = Din .* (kappa_hat .^ (-1 ./ (LT * ones(1, N))))
        
        # Calculating change in prices
        phat = zeros(J, N)
        for j = 1:J
            for n = 1:N
                phat[j, n] = Din_k[n+(j-1)*N, :] * 
                            ((lambda_hat[j, :] .^ (gamma[j, :] ./ T[j])) .* 
                             (c[j, :] .^ (-1/T[j])))'
                phat[j, n] = phat[j, n] ^ (-T[j])
            end
        end
        
        pfdev = abs.(phat .- pf0)  # checking tolerance
        pf0 = phat
        pfmax = maximum(pfdev)
        it += 1
    end
    
    return pf0, c
end

P_h_om (generic function with 1 method)

# GMCnew.m

In [22]:
function GMCnew(Xp, Dinp, J, N, B, gamma, Ljn_hat, VARjn0, VALjn0, R)
    # Calculating new wages using the labor market clearing condition
    PQ_vec = reshape(Xp', J*N, 1)
    
    DP = zeros(J*N, N)
    for n = 1:N
        DP[:, n] = Dinp[:, n] .* PQ_vec
    end
    
    Exjnp = zeros(J, N)
    for j = 1:J
        for n = 1:N
            Exjnp[j, n] = sum(DP[1+N*(j-1):N*j, n])'
        end
    end
    
    aux4 = gamma .* Exjnp
    aux5 = aux4
    omef0 = ones(J, N)
    
    omef0[:, 1:R] = aux5[:, 1:R] ./ 
                    ((Ljn_hat[:, 1:R] .^ (1 .- B[:, 1:R])) .* 
                    (VARjn0[:, 1:R] .+ VALjn0[:, 1:R]))
    
    VAR = sum(VARjn0, dims=1)'
    VAL = sum(VALjn0, dims=1)'
    aux5_sum = sum(aux4, dims=1)'
    omef0[:, R+1:N] = ones(J, 1) * (aux5_sum[R+1:N] ./ (VAR[R+1:N] .+ VAL[R+1:N]))'
    
    return omef0
end

GMCnew (generic function with 1 method)

# solvewnew.m

In [23]:
function solvewnew(om, Ljn_hat, VARjn0, VALjn0, Din, Snp, kappa_hat, lambda_hat, 
    alphas, io, T, B, G, gamma, J, N, maxit, tol, R, vfactor)

ommax = 1
itw = 1

while (itw <= maxit) && (ommax > tol)
# Calculating good prices and input bundle consistent with factor prices
phat, c = P_h_om(om, kappa_hat, lambda_hat, T, B, G, gamma, Din, J, N, maxit, tol)

# Calculating bilateral trade shares
Dinp = Dinprime(Din, kappa_hat, lambda_hat, c, phat, T, J, N, gamma)

# Calculating total expenditure
Xp = expenditurenew(J, N, alphas, B, G, Dinp, om, Ljn_hat, Snp, VARjn0, VALjn0, io)

# Calculating new wages using the factor market clearing condition
omef0 = GMCnew(Xp, Dinp, J, N, B, gamma, Ljn_hat, VARjn0, VALjn0, R)

# Excess function
ZW = om .- omef0

# Iteration factor prices
om1 = om .* (1 .+ vfactor .* ZW ./ om)
om11 = reshape(om1, N*J, 1)
om00 = reshape(om, N*J, 1)
omusa = om11[1:J*R, 1] .- om00[1:J*R, 1]
omrow = om1[1, R+1:N]' .- om[1, R+1:N]'
omworld = [omusa; omrow]
ommax = sum(abs.(omworld .^ 2))  # checking tolerance
ommax0 = ommax
om = om1
itw += 1
end

# Recovering wages and rental rates
wf0 = zeros(J, N)
rf0 = zeros(J, N)
wf0[:, 1:R] = om[:, 1:R] .* (Ljn_hat[:, 1:R] .^ (-B[:, 1:R]))
wf0[:, R+1:N] = om[:, R+1:N]
rf0[:, 1:R] = wf0[:, 1:R] .* Ljn_hat[:, 1:R]
rf0[:, R+1:N] = om[:, R+1:N]

# Recovering deficits
VARjnp = VARjn0 .* om .* (Ljn_hat .^ (1 .- B))
VARp = sum(VARjnp, dims=1)'
Chip = sum(VARp)
Bnp = Snp .- io .* Chip .+ VARp

# Recovering xbilat
PQ = Xp
PQ_vec = reshape(PQ', J*N, 1)
xbilatp = (PQ_vec * ones(1, N)) .* Dinp

# Recovering other variables
VALjnp = wf0 .* Ljn_hat .* VALjn0
VAjnp = VALjnp .+ VARjnp
VAjnp = VAjnp[:, 1:R]

Phat = prod(phat .^ alphas, dims=1)  # price index

return om, wf0, VARjnp, VALjnp, Phat, rf0, phat, Dinp, Xp, Snp, xbilatp
end



function Dinprime(Din, kappa_hat, lambda_hat, c, phat, T, J, N, gamma)
# Implement this based on your MATLAB version
return zeros(size(Din))
end

function expenditurenew(J, N, alphas, B, G, Dinp, om, Ljn_hat, Snp, VARjn0, VALjn0, io)
# Implement this based on your MATLAB version
return zeros(N, N)
end



expenditurenew (generic function with 1 method)

# Dinprime.m

In [24]:
function Dinprime(Din, kappa_hat, lambda_hat, c, phat, T, J, N, gamma)
    # Reformating theta vector
    LT = zeros(J*N, 1)
    for j = 1:J
        idx = 1+(j-1)*N:N*j
        LT[idx, 1] = ones(N) * T[j]
    end
    
    # Calculating bilateral trade shares
    cp = zeros(J, N)
    phatp = zeros(J, N)
    for n = 1:N
        cp[:, n] = c[:, n] .^ (-1 ./ T)
        phatp[:, n] = phat[:, n] .^ (-1 ./ T)
    end
    
    Din_k = Din .* (kappa_hat .^ (-1 ./ (LT * ones(1, N))))
    
    DD = zeros(size(Din))
    for n = 1:N
        idx = n:N:length(Din)-(N-n)
        DD[idx, :] = Din_k[idx, :] .* 
                    (cp .* (lambda_hat .^ (gamma ./ (T * ones(1, N)))))
    end
    
    Dinp = zeros(size(Din))
    for n = 1:N
        idx = n:N:length(Din)-(N-n)
        Dinp[idx, :] = DD[idx, :] ./ (phatp[:, n] * ones(1, N))
    end
    
    return Dinp
end

Dinprime (generic function with 1 method)

# expenditurenew.m

In [25]:

function expenditurenew(J, N, alphas, B, G, Dinp, om, Ljn_hat, Snp, VARjn0, VALjn0, io)
    # Calculate VARjnp and related variables
    VARjnp = VARjn0 .* om .* (Ljn_hat .^ (1 .- B))
    VARp = sum(VARjnp, dims=1)'
    Chip = sum(VARp)
    Bnp = Snp .- io .* Chip .+ VARp
    
    # Calculating the Omega matrix
    NBP = zeros(size(Dinp'))
    
    for j = 1:N
        for n = 1:N
            NBP[j, 1+(n-1)*J:n*J] = Dinp[n:N:J*N, j]
        end
    end
    
    NNBP = kron(NBP, ones(J, 1))
    GG = kron(ones(1, N), G)
    GP = GG .* NNBP
    
    OM = I - GP  # I is the identity matrix from LinearAlgebra
    
    # Calculating total expenditures
    aux = sum(om .* (Ljn_hat .^ (1 .- B)) .* (VARjn0 .+ VALjn0), dims=1)' .- Bnp
    aux2 = kron(aux, ones(J, 1))
    X = inv(OM) * (reshape(alphas, N*J, 1) .* aux2)
    Xp = reshape(X, J, N)
    
    return Xp
end

expenditurenew (generic function with 1 method)

# Base Year .m

In [26]:
# Trade and Labor Market Dynamics: "General Equilibrium Analysis of the China
# Trade Shock, by Caliendo, Dvorkin, and Parro (Econometrica)"

# This Julia file computes the equilibrium allocations at the initial
# period (year 2000)

using DelimitedFiles

# Clear everything (not needed in Julia as we can just restart the kernel)

# Parameters
vfactor = -0.05
tol = 1e-7
maxit = 1e20

# Timing start
t0 = time()

# Inputs
J = 22    # sectors 
JNT = 9   # non-tradables
JT = 13   # tradables
R = 50    # regions
C = 37    # countries
N = R + C # total countries and regions

# dispersion of productivities
T = [1/2.55, 1/5.56, 1/9.27, 1/51.08, 1/4.75, 1/1.66, 1/2.76, 1/6.78,
     1/1.52, 1/11.70, 1/1.01, 1/5.00, 1/4.55, (1/4.55)*ones(9)]

# DATA
B_usa = readdlm(caliendo_path_base_year * "B.txt")      # share of capital structures in value added
B_row = readdlm(caliendo_path_base_year * "B_row.txt")
gamma = readdlm(caliendo_path_base_year * "gamma.txt")  # share of value added in gross production
IO_data = readdlm(caliendo_path_base_year * "IO_tables.txt")  # input-output table
xbilat = readdlm(caliendo_path_base_year * "xbilat.txt")      # bilateral trade flows 2000
GO = readdlm(caliendo_path_base_year * "GO.txt")              # Gross output

# Assuming there's a data function to be defined separately
function data(J, JNT, JT, R, N, C, B_usa, B_row, gamma, IO_data, GO, xbilat)
    # This would need to be implemented based on your MATLAB data function
    # Returning dummy values for now
    B = zeros(J, N)
    gamma = zeros(J)
    G = zeros(J, J)
    Din = zeros(J*N, N)
    VALjn0 = zeros(J, N)
    VARjn0 = zeros(J, N)
    alphas = zeros(J)
    VAR = zeros(J, N)
    Bn = zeros(J, N)
    return B, gamma, G, Din, VALjn0, VARjn0, alphas, VAR, Bn
end

# Call data function
B, gamma, G, Din, VALjn0, VARjn0, alphas, VAR, Bn = 
    data(J, JNT, JT, R, N, C, B_usa, B_row, gamma, IO_data, GO, xbilat)

# Iotas
Chi = sum(VAR, dims=1)
io = (VAR .- Bn) ./ Chi
Sn = Bn .- VAR .+ io .* Chi

# Shocks
kappa_hat = ones(J*N, N)    # relative change in trade costs
lambda_hat = ones(J, N)     # relative change in technology
Snp = zeros(size(Sn))

# Initialize vectors of factor prices (w,r) and good prices (p)
L_hat = ones(N)
pf0 = ones(J, N)
ommax = 1
itw = 1
Ljn_hat = ones(J, N)
Ljn_hat[:, R+1:N] .= 1    # Ljn_hat RoW must be one
om = ones(J, N)


# Solve equilibrium
om, wf0, VARjnp, VALjnp, Phat, rf0, phat, Dinp, Xp, Snp, xbilatp = 
    solvewnew(om, Ljn_hat, VARjn0, VALjn0, Din, Snp, kappa_hat, lambda_hat, 
             alphas, io, T, B, G, gamma, J, N, maxit, tol, R, vfactor)

VALjn00 = VALjnp
VARjn00 = VARjnp
Sn00 = Snp
Din00 = Dinp
xbilat00 = xbilatp

# To save data in Julia, you could use something like:
# using JLD2
# @save "Base_year.jld2" VALjn00 VARjn00 Sn00 Din00 xbilat00

LoadError: BoundsError: attempt to access 22-element Vector{Float64} at index [1:22, 2]

caliendo